# AI Engineer 13주차 (정채진 강사님) — 내풀이 v1

정답지 미참조. 문제지 + broken 파일 기준 풀이.

⚠️ 스크린샷/실제 AWS·GCP 콘솔 증빙이 요구되는 항목은 `[SCREENSHOT REQUIRED]` 플레이스홀더로 표시.

---
## 문제 1-1. Backend Dockerfile (5점)

In [ ]:
%%writefile Dockerfile.backend
FROM python:3.11-slim

# (L4-5) healthcheck용 curl 설치 + apt 캐시 정리
RUN apt-get update && apt-get install -y --no-install-recommends curl \
    && rm -rf /var/lib/apt/lists/*

# (L8) 작업 디렉토리
WORKDIR /app

# (L11) uv 설치
RUN pip install --no-cache-dir uv

# (L14) 의존성 파일만 선복사
COPY pyproject.toml uv.lock ./

# (L17) 의존성 설치 (캐시 레이어로 고정)
RUN uv sync --frozen

# (L20) 소스 코드 복사
COPY . .

EXPOSE 8000

CMD ["uv", "run", "uvicorn", "app:app", "--host", "0.0.0.0", "--reload", "--port", "8000"]

#### 서술: 의존성 선복사 이유

위 Dockerfile의 **L14(`COPY pyproject.toml uv.lock ./`)**, **L17(`RUN uv sync --frozen`)**, **L20(`COPY . .`)** 순서가 핵심이다. Docker는 각 명령어 결과를 레이어로 캐시하고, 상위 레이어 중 하나라도 바뀌면 하위 레이어를 모두 재빌드한다. 의존성 파일은 드물게 변경되지만 소스 코드(`app.py` 등)는 매번 변경되므로, L14/L17을 L20보다 앞에 두면 소스만 수정된 빌드에서 L17(의존성 설치)이 캐시에서 재사용되어 빌드 시간이 수십 초에서 수 초로 단축된다.

#### `docker build` 스크린샷

`[SCREENSHOT REQUIRED]` — `date && hostname` 출력 포함 필수

---
## 문제 1-2. docker-compose 수정 (4점)

In [ ]:
%%writefile docker-compose-fixed.yml
services:
  backend:
    build: ./backend
    ports:
      - "1234:8000"
    env_file:
      - ./backend/.env.production
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    networks:
      - agent-network

  frontend:
    build: ./frontend
    ports:
      - "1235:80"
    depends_on:
      backend:
        condition: service_healthy
    networks:
      - agent-network

networks:
  agent-network:
    driver: bridge

#### 오류 설명

**오류 1** — healthcheck 포트
> 잘못된 값: `http://localhost:3000/health`
> 올바른 값: `http://localhost:8000/health`
> 이유: backend 컨테이너 내부 uvicorn이 8000에서 listen하는데 3000을 체크하면 connection refused로 항상 unhealthy.

**오류 2** — backend의 networks 이름
> 잘못된 값: `backend.networks: [app-network]`
> 올바른 값: `[agent-network]`
> 이유: 최상단 networks 섹션에 `app-network`가 정의되지 않음. frontend는 `agent-network`를 쓰는데 backend만 다른 네트워크를 참조하면 서비스명 DNS 해석 불가.

**오류 3** — frontend의 depends_on 조건 누락
> 잘못된 값: `depends_on: [backend]`
> 올바른 값: `depends_on: backend: condition: service_healthy`
> 이유: 단순 리스트는 컨테이너 기동 순서만 보장하고 healthy 여부는 무시. backend에 healthcheck가 정의돼 있는데 이를 활용하지 않으면 backend 부팅 중인 상태에서 frontend가 요청 보내 502 발생.

**오류 4** — obsolete `version` 속성
> 잘못된 값: `version: "3.8"` (top-level)
> 올바른 값: 해당 줄 삭제
> 이유: Compose V2부터 top-level `version`은 무시되며 경고 출력. 최신 스펙에 맞춰 제거.

#### 스크린샷
`[SCREENSHOT REQUIRED]` — `docker compose up`으로 두 서비스 healthy 확인

---
## 문제 1-3. 보안 사고 대응 (3점)

### (a) 3단계 긴급 대응 절차

1. **OpenAI 대시보드에서 노출된 키 즉시 revoke/delete** — 추가 과금 및 오용 차단이 최우선. Git 삭제보다 먼저.
2. **새 API 키 발급 후 AWS Secrets Manager(또는 GCP Secret Manager)에 저장** — `.env` 평문 저장 금지. `.gitignore`에 `.env` 추가하고 `.env.example`만 커밋.
3. **git history에서 파일 완전 제거 후 force push** — GitHub은 push된 커밋을 인덱싱하므로 history 정리가 필수.

### (b) "새 커밋으로 삭제하면 되나요?"

안 된다. HEAD에서만 사라질 뿐 과거 커밋에는 그대로 남아있다. 다음 명령으로 누구나 키 추출 가능:
```bash
# 현재 디렉토리에 .env가 없어도 과거 내용 조회 가능
git log --all --full-history -- .env
git show <commit>:.env
```
GitHub public repo의 경우 crawler/secret scanner(trufflehog, gitleaks)가 이미 키를 수집했을 가능성이 매우 높아, revoke가 답이다.

### (c) git history 완전 제거

```bash
# 추천: git filter-repo (최신, 빠름)
pip install git-filter-repo
git filter-repo --path .env --invert-paths

# 대안: bfg
bfg --delete-files .env

# 정리 후 강제 푸시
git push --force --all
git push --force --tags
```

---
## 문제 2-1. nginx.conf (5점)

In [ ]:
%%writefile nginx.conf
events {
    worker_connections 1024;
}

http {
    include       /etc/nginx/mime.types;
    default_type  application/octet-stream;

    upstream backend {
        server backend:8000;
    }

    server {
        listen 80;
        server_name _;

        # 헬스체크
        location /health {
            access_log off;
            return 200 "OK\n";
        }

        # API 프록시 — /api/ prefix 제거하여 backend에 전달
        location /api/ {
            proxy_pass http://backend/;   # 끝의 슬래시로 /api/ prefix 제거됨

            proxy_http_version 1.1;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto $scheme;
            proxy_set_header Connection '';

            # SSE 스트리밍 지원
            proxy_buffering off;
            proxy_cache off;
            chunked_transfer_encoding on;

            proxy_read_timeout 300s;
        }

        # SPA 정적 파일 서빙
        location / {
            root /usr/share/nginx/html;
            try_files $uri $uri/ /index.html;
        }
    }
}

#### 스크린샷
`[SCREENSHOT REQUIRED]` — `curl http://localhost:1235/api/ask?q=hello` 응답이 backend 8000 로그에 `/ask?q=hello`로 도착하는지 확인

---
## 문제 2-2. 502 디버깅 (4점)

### (a) 원인 3가지 (확률 높은 순)

1. **backend 컨테이너가 not running** — 크래시 또는 healthcheck 실패로 stop된 상태. nginx가 upstream에 TCP 연결 실패.
2. **Docker 내장 DNS 해석 실패** — nginx의 `proxy_pass http://backend:8000/`에서 `backend` 서비스명이 같은 네트워크(`agent-network`)에 없어 NXDOMAIN.
3. **backend가 잘못된 인터페이스에 bind** — uvicorn이 `127.0.0.1:8000`에 바인드하면 컨테이너 외부(nginx 측)에서 접근 불가. `0.0.0.0:8000`이어야 함.

### (b) 프로젝트 특화 디버깅 명령

In [ ]:
# 원인 1 검증: backend 컨테이너 상태
!docker compose ps backend
!docker compose logs --tail=100 backend

# 원인 2 검증: nginx 컨테이너에서 DNS 해석
!docker network inspect agent-network --format '{{range .Containers}}{{.Name}} {{.IPv4Address}}\n{{end}}'
!docker exec -it $(docker compose ps -q frontend) getent hosts backend
!docker exec -it $(docker compose ps -q frontend) curl -v http://backend:8000/health

# 원인 3 검증: backend의 listen 인터페이스
!docker exec -it $(docker compose ps -q backend) ss -tlnp
# 기대: 0.0.0.0:8000 (127.0.0.1:8000이면 원인 3 확정)

### (c) Docker Compose bridge 네트워크에서 서비스명 통신 원리

Compose가 `agent-network`(bridge) 생성 시 **127.0.0.11에 embedded DNS 서버**를 내장한다. 동일 네트워크에 속한 모든 컨테이너는 `/etc/resolv.conf`의 nameserver로 이 주소를 사용하도록 자동 구성된다. 컨테이너가 `backend`를 쿼리하면 DNS가 **서비스명 → 컨테이너 IP**로 해석해주고, bridge 네트워크이므로 L2 프레임이 직접 전달된다. 다른 네트워크나 호스트 네임스페이스에 있는 컨테이너는 이 DNS에 등록되지 않아 해석 불가.

---
## 문제 2-3. 헬스체크 설정 (3점)

### (a) healthcheck YAML

In [ ]:
healthcheck_yaml = """
healthcheck:
  test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
  interval: 30s
  timeout: 10s
  retries: 3
  start_period: 40s
"""
print(healthcheck_yaml)

### (b) 파라미터 역할

| 파라미터 | 설명 |
|----------|------|
| `interval` | 헬스체크 실행 주기. 30s면 30초마다 test 명령 수행. |
| `timeout` | 한 번의 test 명령이 응답을 기다리는 최대 시간. 초과 시 fail 1회. |
| `retries` | 연속 fail이 이 횟수에 도달하면 컨테이너 상태가 `unhealthy`로 전환. |
| `start_period` | 컨테이너 시작 후 이 구간 동안의 fail은 `retries` 카운트에 포함하지 않는 grace period. 애플리케이션 warm-up 시간 보장. |

### (c) 시나리오 — `start_period=0`, 앱 기동 30초

- **t=0s**: 컨테이너 start. start_period=0이므로 즉시 healthcheck 시작.
- **t=0s**: 첫 test 실행 → 앱 아직 부팅 중 → curl connection refused → fail 1 (retries=1 카운트).
- **t=10s**: timeout 만료.
- **t=30s**: 두 번째 test → 여전히 부팅 완료 전일 수 있음 → fail 2.
- **t=60s**: 세 번째 test → fail 3 → `unhealthy` 확정.

frontend는 `depends_on.backend.condition: service_healthy`에 의해 backend가 healthy가 될 때까지 시작되지 않는다. backend가 `unhealthy`로 낙인 찍히면 frontend는 영원히 대기하고, `docker compose up`은 `dependency failed to start` 에러로 종료된다. **따라서 앱 부팅 시간보다 넉넉한 `start_period`(예: 40~60s) 설정이 필수.**

---
## 문제 3-1. Dual ALB 아키텍처 다이어그램 (4점)

```
                             [ Internet / Client ]
                                      │
                              HTTPS:443, HTTP:80
                                      │
                         ┌────────────┴──────────────┐
                         │ Frontend ALB SG           │
                         │ inbound : 80,443 from     │
                         │          0.0.0.0/0         │
                         │ outbound: all             │
                         └────────────┬──────────────┘
                                      │
                          [ Frontend ALB :80/:443 ]
                                      │
                                      ▼
                     [ Frontend Target Group :80 ]
                                      │
                         ┌────────────┴──────────────┐
                         │ Frontend Container SG     │
                         │ inbound : 80 from         │
                         │          Frontend-ALB-SG │
                         │ outbound: all             │
                         └────────────┬──────────────┘
                                      │
                    [ Fargate Task: nginx:80 ]
                                      │
                  proxy_pass https://backend-alb/
                                      │
                                      ▼
                         ┌─────────────────────────────┐
                         │ Backend ALB SG              │
                         │ inbound : 80 from           │
                         │          Frontend-Container-SG
                         │ outbound: all               │
                         └────────────┬────────────────┘
                                      │
                          [ Backend ALB :80 (internal) ]
                                      │
                                      ▼
                     [ Backend Target Group :8000 ]
                                      │
                         ┌────────────┴──────────────┐
                         │ Backend Container SG      │
                         │ inbound : 8000 from       │
                         │          Backend-ALB-SG  │
                         │ outbound: all             │
                         └────────────┬──────────────┘
                                      │
                 [ Fargate Task: FastAPI :8000 ]

SSE 역방향 흐름 (토큰 스트리밍):
 FastAPI:8000 ──▶ Backend ALB ──▶ nginx(frontend) ──▶ Frontend ALB ──▶ Client
 (chunked transfer, proxy_buffering off 유지)
```

`[EXCALIDRAW SOURCE FILE REQUIRED]` — `.excalidraw`로 저장 후 제출

`[SCREENSHOT REQUIRED]` — AWS ECS 콘솔 Services 탭에서 RUNNING 상태 증빙

---
## 문제 3-2. Task Definition 수정 (4점)

### 오류 설명

**오류 1** — executionRoleArn
> 잘못된 값: `arn:aws:iam::123456789:role/wrongRoleName`
> 올바른 값: `arn:aws:iam::123456789:role/ecsTaskExecutionRole-dev`
> 이유: executionRole은 ECR pull, CloudWatch Logs push, Secrets Manager fetch 등 ECS 에이전트가 쓰는 role이어야 한다. 임의 이름이 아닌 `AmazonECSTaskExecutionRolePolicy`가 부착된 표준 role.

**오류 2** — secrets 배열의 `name` 키 중복
> 잘못된 값: `{"name": "OPENAI_API_KEY", "name": "openai-api-key-dev"}`
> 올바른 값: `{"name": "OPENAI_API_KEY", "valueFrom": "arn:aws:secretsmanager:ap-northeast-2:123456789:secret:openai-api-key-dev-xxxx"}`
> 이유: ECS secrets 스펙은 `name`(컨테이너 환경변수명) + `valueFrom`(Secrets Manager ARN 또는 SSM Parameter name) 쌍이어야 한다. `name`이 두 번 있으면 JSON 파싱 시 뒤 값이 앞을 덮어써 환경변수명 자체가 `openai-api-key-dev`가 되어 버림.

**오류 3** — awslogs-group
> 잘못된 값: `/ecs/wrong-log-group`
> 올바른 값: `/ecs/backend-dev`
> 이유: Log group은 사전에 Terraform으로 생성되어야 하며 family(`backend-dev`)와 매칭되는 관례적 이름을 사용한다. 존재하지 않는 group명 지정 시 awslogs driver가 `ResourceNotFoundException` 발생.

### 서술: executionRoleArn vs taskRoleArn

**executionRoleArn**은 ECS 에이전트(fargate platform 자체)가 **태스크를 기동·운영하는 시점**에 사용하는 IAM role이다. 구체적으로 ECR로부터 이미지를 pull하고, CloudWatch Logs로 stdout을 전송하며, Secrets Manager/SSM Parameter Store에서 secrets를 가져와 컨테이너에 환경변수로 주입할 때 호출된다. 이 role이 없거나 권한이 부족하면 태스크 자체가 시작되지 못한다. 

**taskRoleArn**은 컨테이너 내부에서 실행되는 **애플리케이션 코드가 런타임에 AWS SDK로 다른 서비스를 호출할 때** 사용하는 role이다. 예를 들어 FastAPI 핸들러가 boto3로 S3에 파일 업로드, DynamoDB에 쓰기, SQS로 메시지 전송할 때 이 role의 권한을 부여받는다. EC2의 instance profile과 개념적으로 동일하다.

즉 전자는 **플랫폼 레벨(태스크 생명주기 관리)**, 후자는 **애플리케이션 레벨(런타임 AWS API 호출)**에서 작동하며, 최소 권한 원칙에 따라 서로 다른 role에 서로 다른 policy를 부착하는 것이 표준이다.

---
## 문제 3-3. Health Check 설계 (4점)

### (a) Health Check 3단계 실패 시 동작

| 레벨 | 설정 위치 | 실패 시 동작 | 확인 방법 |
|---|---|---|---|
| 컨테이너 | Task Definition `healthCheck` (Docker HEALTHCHECK) | Docker가 컨테이너를 `unhealthy` 표시 → ECS가 해당 Task를 STOPPED 처리 후 Service가 desiredCount 유지 위해 새 Task 기동 | `aws ecs describe-tasks --cluster agent-cluster-prod --tasks <taskId> --query 'tasks[0].containers[0].healthStatus'` |
| ALB Target Group | Target Group `health_check` | unhealthy target으로 마킹 → ALB가 트래픽 라우팅에서 제외, deregistration_delay 후 제거. ECS Service와 연동 시 해당 Task는 교체 대상 | `aws elbv2 describe-target-health --target-group-arn <tg-arn>` 또는 콘솔 Target Group → Targets 탭 |
| 애플리케이션 | FastAPI `/health`, `/ready` 엔드포인트 | 500/503 반환 → 위 두 레벨이 연쇄적으로 실패 판정 → Task 교체 | `curl http://<task-private-ip>:8000/health` (VPC 내부에서) 또는 ALB 경유 `curl https://backend-alb/health` |

### (b) ALB Target Group Health Check HCL

In [ ]:
%%writefile health-check.tf
resource "aws_lb_target_group" "backend" {
  name        = "backend-tg-${var.environment}"
  port        = 8000
  protocol    = "HTTP"
  vpc_id      = var.vpc_id
  target_type = "ip"

  health_check {
    enabled             = true
    path                = "/health"
    port                = "traffic-port"
    protocol            = "HTTP"
    healthy_threshold   = 2
    unhealthy_threshold = 3
    interval            = 30
    timeout             = 10
    matcher             = "200"
  }

  deregistration_delay = 30

  tags = {
    Environment = var.environment
  }
}

### (c) `/health` vs `/ready`

- **`/health` (Liveness)**: 프로세스가 살아있는지만 확인. 단순히 HTTP 200을 즉시 반환. 외부 의존성 체크는 하지 않는다. 실패 시 **컨테이너 재시작 대상**.
- **`/ready` (Readiness)**: 트래픽을 받을 수 있는 상태인지 확인. DB connection pool 준비, LLM API 핸드셰이크, 벡터 DB 인덱스 로드 완료, 모델 warm-up 종료 등 외부 의존성을 포함해서 체크. 실패 시 **트래픽 라우팅에서 제외하되 재시작은 하지 않음** (외부 의존성은 잠시 후 복구될 수 있음).

프로젝트 `app.py`에서 `/health`는 `return {"status": "ok"}`로 즉답, `/ready`는 `try: vector_db.ping(); llm.test_connection(); return {"ready": True}`처럼 의존성 체크 로직을 수행. **ALB Target Group은 `/health`, Kubernetes readinessProbe 또는 ECS rolling deployment의 gate 용도로는 `/ready`**를 사용.

---
## 문제 4-1. AWS vs GCP 비교 (4점)

| 비교 항목 | AWS ECS/Fargate | GCP Cloud Run | 더 간단한 쪽 + 이유 |
|-----------|-----------------|---------------|---------------------|
| 이미지 레지스트리 | **ECR**<br>`aws ecr get-login-password --region ap-northeast-2 \| docker login --username AWS --password-stdin 123.dkr.ecr.ap-northeast-2.amazonaws.com`<br>`docker push <account>.dkr.ecr.ap-northeast-2.amazonaws.com/backend:v1` | **Artifact Registry**<br>`gcloud auth configure-docker asia-northeast3-docker.pkg.dev`<br>`docker push asia-northeast3-docker.pkg.dev/<proj>/backend/backend:v1` | 비슷 — 둘 다 docker login 후 push 동일 패턴 |
| 컨테이너 오케스트레이션 | ECS Cluster + Task Definition(JSON) + Service 3개 리소스 관리 | `gcloud run deploy` 한 줄로 컨테이너 배포 완료 | **GCP** — Fargate는 Task Def/Service 분리, Cloud Run은 단일 서비스 추상화 |
| 로드밸런싱 | ALB + Listener + Target Group 수동 구성, DNS도 직접 설정 | 자동 내장, 배포 즉시 `https://<service>-<hash>-<region>.a.run.app` 할당 | **GCP** — LB 설정 자체가 없음 |
| 시크릿 관리 | Secrets Manager + TaskDef `secrets.valueFrom` | Secret Manager + `gcloud run deploy --set-secrets=KEY=secret-name:version` | **GCP** — 플래그 하나로 주입 완료 |
| 로깅 | CloudWatch Logs (log group 사전 생성 필요, awslogs driver 설정) | Cloud Logging 자동 수집 (stdout/stderr) | **GCP** — 설정 제로 |
| 오토스케일링 | Application Auto Scaling Target + Policy 설정 | `--min-instances`, `--max-instances` 플래그 | **GCP** — 요청 수 기반 자동 스케일 |
| Cold Start | 없음 (desiredCount≥1이면 상시 기동) | 기본 있음 (scale-to-zero 기본값) | **AWS — latency 측면**, **GCP — 비용 측면** (요건에 따라 trade-off) |
| 비용 모델 | vCPU × 시간 + GB × 시간, 기본적으로 **상시 과금** | 요청 처리 시간 × 할당 CPU·Memory, **scale-to-zero로 idle 시 $0** | **GCP 저트래픽**, **AWS 고트래픽 상시**

---
## 문제 4-2. Cloud Run용 nginx.conf 수정 (3점)

In [ ]:
%%writefile nginx-cloudrun.conf
location /api/ {
    # (a) HTTPS로 변경 — Cloud Run은 HTTP 포트 비공개
    proxy_pass https://backend-service-xxxxxx-an.a.run.app/;

    proxy_http_version 1.1;

    # (b) Host 헤더 필수 — Cloud Run은 Host 기반 SNI 라우팅
    proxy_set_header Host backend-service-xxxxxx-an.a.run.app;
    proxy_ssl_server_name on;
    proxy_set_header X-Forwarded-Proto https;

    # SSE (기존과 동일)
    proxy_buffering off;
    proxy_cache off;
    chunked_transfer_encoding on;
    proxy_set_header Connection '';

    # (c) Cloud Run 인증이 필요하면 IAM ID Token 주입
    # proxy_set_header Authorization "Bearer $gcp_id_token";

    # (c) LLM 스트리밍용 timeout 연장
    proxy_read_timeout 300s;
    proxy_send_timeout 300s;
}

### 3가지 핵심 차이점

**(a) HTTPS vs HTTP**
Cloud Run은 외부 노출 포트가 **HTTPS(443)만** 열려 있다. `http://` 스킴으로 `proxy_pass`하면 HTTP→HTTPS 리다이렉트(302/307)를 받아 실제 API 응답을 얻지 못한다. 반드시 `proxy_pass https://...`로 작성.

**(b) `proxy_set_header Host` 설정**
Cloud Run은 **도메인 Host 헤더로 서비스를 라우팅**한다(SaaS 멀티테넌시 방식, SNI). Docker Compose bridge 환경에서는 DNS가 IP를 주고 TCP 직결이므로 Host가 의미 없었지만, Cloud Run은 nginx → Cloud Front-end LB로 가는 요청이 정확히 해당 서비스의 도메인을 Host로 가져야 식별된다. 빠뜨리면 `404 Service Not Found`.

**(c) `proxy_read_timeout` 증가**
nginx 기본 `proxy_read_timeout`은 60초. Cloud Run 자체는 request timeout을 최대 60분(3600초)까지 지원한다. LLM이 긴 답변을 토큰 스트리밍할 때 60초를 넘는 경우가 흔하므로, nginx 레벨에서 먼저 끊어지지 않게 최소 300초 이상으로 연장 필요.

---
## 문제 4-3. Cold Start 분석 (3점)

### (a) 8초간의 컨테이너 초기화 프로세스

1. **이미지 pull** (1~3s) — Artifact Registry에서 컨테이너 이미지 레이어 다운로드. 이미지 크기에 비례. 수백 MB 이미지면 2초 내외.
2. **컨테이너 unpack & mount** (~1s) — Docker 런타임이 레이어 추출, overlayfs mount, rootfs 준비.
3. **프로세스 기동** (~0.5s) — Cloud Run이 지정된 CMD(`uvicorn app:app ...`) 실행, Python 인터프리터 startup.
4. **애플리케이션 초기화** (3~5s) — FastAPI import, LangChain/LangGraph 그래프 빌드, tokenizer 로드, 벡터 DB 클라이언트 연결, 임베딩 모델 사전 로드.
5. **Listen 시작** (~0.1s) — uvicorn이 `0.0.0.0:8080`(Cloud Run은 PORT env) 바인딩.
6. **첫 요청 처리** (~0.5~1s) — Cloud Run probe가 listen 감지 후 사용자 요청 라우팅, OpenAI API로 첫 핸드셰이크.

### (b) Cold Start 감소 전략

| 전략 | 효과 | 트레이드오프 |
|------|------|-------------|
| `--min-instances 1` | Cold start 완전 제거 (항상 1대 warm) | scale-to-zero 포기 → 월 상시 $10~30 |
| `--cpu-boost` | 부팅 시 CPU 2배 할당 → 4단계 초기화 속도 2배 | 부팅 시점 vCPU 사용량 증가, 약간의 비용 증가 |
| 멀티스테이지 Dockerfile + `python:3.11-slim` | 이미지 크기 축소 → 1단계 pull 시간 단축 | 빌드 파이프라인 복잡도 증가, 시스템 패키지 누락 위험 |

### (c) `min-instances=1` 선택 시점

다음 중 하나라도 해당되면 적극 고려:
- 사용자 대면 실시간 서비스로 **P99 latency SLO < 2초** 요구
- 트래픽이 완전히 0이 되지 않고 **시간당 최소 수 건** 이상 꾸준함 (warm 유지 효율 ↑)
- 첫 응답 지연이 UX에 치명적 (챗봇 첫 메시지, 결제 API)
- **월 상시 비용(10~30 USD)이 비즈니스 임팩트보다 작음** — cold start 1건이 이탈로 이어질 때 기대 손실 > 상시 비용
- LangChain/LLM 그래프 빌드가 무거워 cold start 실측 > 5초

---
## 문제 5-1. Security Group Terraform (5점)

In [ ]:
%%writefile security-groups.tf
# Backend ALB Security Group
resource "aws_security_group" "backend_alb" {
  name        = "backend-alb-sg-${var.environment}"
  description = "Security group for Backend ALB"
  vpc_id      = var.vpc_id

  ingress {
    description = "HTTP from internet"
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    description = "HTTPS from internet"
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    description = "All outbound"
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name        = "backend-alb-sg-${var.environment}"
    Environment = var.environment
  }
}

# Backend Container (Fargate Task) Security Group
resource "aws_security_group" "backend_container" {
  name        = "backend-container-sg-${var.environment}"
  description = "Security group for Backend Fargate Task"
  vpc_id      = var.vpc_id

  ingress {
    description     = "App port 8000 from Backend ALB only"
    from_port       = 8000
    to_port         = 8000
    protocol        = "tcp"
    security_groups = [aws_security_group.backend_alb.id]
  }

  egress {
    description = "All outbound (OpenAI API, Secrets Manager, ECR, etc.)"
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name        = "backend-container-sg-${var.environment}"
    Environment = var.environment
  }
}

### 서술: Container SG에서 `security_groups` 참조를 쓰는 이유

`cidr_blocks = ["0.0.0.0/0"]`로 열면 Backend Container의 8000 포트가 **VPC 내외 어디서든 접근 가능**해지며 ALB를 우회한 직접 트래픽까지 허용된다. 이는 (1) ALB에만 적용된 WAF·rate limit·TLS termination을 무력화하고 (2) 공격 표면을 확대하며 (3) 로그·메트릭이 ALB를 거치지 않아 관찰 가능성이 떨어진다.

`security_groups = [aws_security_group.backend_alb.id]`로 ALB SG를 참조하면, AWS가 **source SG 기반 필터링**을 적용하여 "이 특정 SG가 attach된 ENI(=ALB의 인터페이스)에서 온 트래픽"만 통과시킨다. IP가 아니라 **리소스 아이덴티티 기반 규칙**이므로 ALB IP가 바뀌거나 subnet이 확장되어도 변경 불필요하고, SG chaining으로 네트워크 경계를 논리적으로 명확히 표현할 수 있어 Defense in Depth의 핵심 패턴이 된다.

---
## 문제 5-2. variables & outputs (4점)

In [ ]:
%%writefile variables.tf
variable "environment" {
  description = "배포 환경 (dev, staging, prod)"
  type        = string

  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment는 dev, staging, prod 중 하나여야 합니다."
  }
}

variable "vpc_id" {
  description = "배포 대상 VPC ID (default 없음 — 환경마다 반드시 명시)"
  type        = string
}

variable "public_subnet_ids" {
  description = "ALB를 배치할 퍼블릭 서브넷 ID 목록 (AZ 2개 이상 권장)"
  type        = list(string)

  validation {
    condition     = length(var.public_subnet_ids) >= 2
    error_message = "ALB 고가용성을 위해 최소 2개의 AZ에 걸친 서브넷을 지정해야 합니다."
  }
}

variable "private_subnet_ids" {
  description = "Fargate Task가 배치될 프라이빗 서브넷 ID 목록"
  type        = list(string)
}

variable "backend_cpu" {
  description = "Backend Fargate Task CPU units (256, 512, 1024, 2048, 4096)"
  type        = string
  default     = "512"
}

variable "backend_memory" {
  description = "Backend Fargate Task Memory (MB)"
  type        = string
  default     = "1024"
}

variable "backend_desired_count" {
  description = "Backend Service의 desired task count"
  type        = number
  default     = 2
}

variable "openai_api_key_secret_arn" {
  description = "OPENAI_API_KEY가 저장된 Secrets Manager ARN"
  type        = string
}

In [ ]:
%%writefile outputs.tf
output "frontend_alb_dns" {
  description = "Frontend ALB DNS 이름 (퍼블릭)"
  value       = aws_lb.frontend.dns_name
}

output "backend_alb_dns" {
  description = "Backend ALB DNS 이름 (internal)"
  value       = aws_lb.backend.dns_name
}

output "ecs_cluster_name" {
  description = "ECS Cluster 이름"
  value       = aws_ecs_cluster.main.name
}

output "backend_service_name" {
  description = "Backend ECS Service 이름"
  value       = aws_ecs_service.backend.name
}

output "backend_log_group_name" {
  description = "Backend CloudWatch Log Group"
  value       = aws_cloudwatch_log_group.backend.name
}

### 서술: vpc_id에 default가 없어야 하는 이유

VPC ID는 **AWS 계정·리전마다 고유한 식별자**이며 환경(dev/staging/prod)마다 반드시 다른 값을 써야 한다. 만약 `default = "vpc-xxx"`를 두면 개발자가 `-var="environment=prod"`만 지정하고 `vpc_id`를 빠뜨릴 경우 **잘못된 VPC(예: dev VPC)에 prod 리소스를 생성**하는 참사가 발생할 수 있다. Terraform은 default가 없는 변수를 **필수 입력**으로 처리하여 값 누락 시 plan/apply가 실패하므로, 안전 장치 역할을 한다. `vpc_id`, `subnet_ids`처럼 환경 특정 식별자는 default 금지가 표준 관행이다.

---
## 문제 5-3. 수동 vs Terraform (5점)

### (a) 수동 재구성 문제점 3가지

1. **휴먼 에러로 인한 구성 드리프트** — 포트/서브넷/SG 규칙/taskDef JSON의 수십 개 파라미터 중 하나라도 오타·누락 시 원본과 미묘하게 달라진다. 장애 상황이라 급하게 진행하면 오히려 신규 장애를 유발할 가능성이 높다.
2. **복구 시간** — ECS cluster·service·task definition·ALB·target group·SG·IAM·CloudWatch·ECR·Route53 등 수십 개 리소스를 콘솔 클릭으로 재현하면 최소 수 시간. 장애 중 MTTR(평균 복구 시간)이 폭증한다.
3. **일관성·검증 불가** — 원본 리전과 새 리전의 구성이 동일한지 확인할 표준 방법이 없다. 콘솔 UI로 비교하려면 각 리소스를 클릭해 수십 번의 스크린샷 대조가 필요하고, diff가 발생해도 추적 곤란.

### (b) Terraform의 해결

동일한 `.tf` 파일을 `-var="region=ap-southeast-1"` 하나만 바꿔 `apply`하면 **완전 동일한 인프라가 수 분~수십 분 내 재현**된다. `terraform plan`으로 예상 변경을 dry-run 검토할 수 있고, `terraform state`로 각 리소스의 상태를 명시적으로 추적한다. `terraform destroy` → 재생성으로 원자적 복구 가능하며, git history로 "언제 누가 무엇을 바꿨는지" audit이 된다.

### (c) 새 리전 배포 4개 명령어

In [ ]:
# 순서가 중요한 이유는 주석 참고

# 1. terraform init
#    - provider 플러그인(aws, tls 등) 다운로드
#    - backend (예: S3) 연결, state 파일 위치 초기화
#    - 이게 선행되지 않으면 plan/apply 자체가 실패
# terraform init \
#   -backend-config="bucket=metacode-terraform-state" \
#   -backend-config="key=envs/prod-sg1/terraform.tfstate" \
#   -backend-config="region=ap-southeast-1"

# 2. terraform plan
#    - 생성/변경/삭제될 리소스를 미리보기 (dry-run)
#    - 예상치 못한 destroy를 apply 전에 반드시 확인
# terraform plan -var-file="prod.tfvars" -out=plan.out

# 3. terraform apply
#    - plan에서 확인한 내용을 실제 AWS에 적용
#    - -auto-approve는 비상복구 상황에서만 (일반적으로는 수동 승인)
# terraform apply plan.out

# 4. terraform output
#    - 생성된 ALB DNS, Cluster name 등을 조회 → Route53/모니터링 연결에 사용
# terraform output

In [ ]:
%%writefile terraform.tfvars
# Production terraform.tfvars — Fargate 유효 CPU/Memory 조합 준수
environment           = "prod"
vpc_id                = "vpc-0abc1234def567890"
public_subnet_ids     = ["subnet-0aa111", "subnet-0bb222", "subnet-0cc333"]
private_subnet_ids    = ["subnet-1aa111", "subnet-1bb222", "subnet-1cc333"]

# CPU 1024 (1 vCPU), Memory 4096 — 유효 조합 (1024 CPU는 2048~8192 MB 지원)
backend_cpu           = "1024"
backend_memory        = "4096"
backend_desired_count = 3

openai_api_key_secret_arn = "arn:aws:secretsmanager:ap-northeast-2:123456789012:secret:openai-api-key-prod-abc123"

**state 파일의 목적**: Terraform이 관리하는 리소스의 현재 상태(리소스ID, 속성값, 종속성)를 저장하는 source of truth. 실제 AWS 상태와 HCL 코드 사이의 매핑 정보. Team 협업 시 원격 backend(S3+DynamoDB lock)에 저장하여 동시성 제어와 상태 공유를 보장한다.

---
## 문제 6-1. GitHub Actions 워크플로우 수정 (5점)

### 오류 설명

**오류 1** — staging 브랜치 누락
> 잘못된 값: `branches: [main, develop]`
> 올바른 값: `branches: [main, develop, staging]`
> 이유: 3-Stage 전략(develop→staging→main)에서 staging 배포가 트리거되지 않음.

**오류 2** — 환경 매핑 반대
> 잘못된 값: `environment: ${{ github.ref_name == 'main' && 'development' || 'production' }}`
> 올바른 값: `environment: ${{ github.ref_name == 'main' && 'production' || (github.ref_name == 'staging' && 'staging' || 'development') }}`
> 이유: main은 production에 배포되어야 한다. 현재 코드는 main을 development로, 나머지(develop, staging)를 production으로 매핑하여 완전히 반대로 되어 있음.

**오류 3** — `-backend-config` key 누락
> 잘못된 값: `terraform init -backend-config="bucket=metacode-terraform-state"`
> 올바른 값: `terraform init -backend-config="bucket=metacode-terraform-state" -backend-config="key=envs/${{ vars.ENVIRONMENT }}/terraform.tfstate" -backend-config="region=${{ vars.AWS_REGION }}"`
> 이유: S3 backend는 최소 bucket/key/region 3개 config가 필요. key(state 파일 경로)가 없으면 init 실패하거나 모든 환경이 같은 state 파일을 공유해 충돌.

**오류 4** — Dockerfile 경로 오류
> 잘못된 값: `docker build -t backend -f ./Dockerfile ./backend`
> 올바른 값: `docker build -t backend -f ./backend/Dockerfile ./backend`
> 이유: `-f` 뒤 Dockerfile 위치는 `./backend/Dockerfile`에 있는데 현재는 레포 루트의 `./Dockerfile`을 참조 → `file not found`.

### 서술: 3-Stage 배포 전략 순서

**ECR 생성 → 이미지 빌드/푸시 → 전체 인프라 배포** 순서가 강제되는 이유:
- ECS Task Definition의 `image` 필드는 **실제로 pull 가능한 이미지 URI**를 가리켜야 하는데, 이를 위해서는 (1) ECR repository가 존재해야 하고 (2) 해당 태그의 이미지가 미리 push되어 있어야 한다.
- 순서가 뒤집히면: ECS Service가 기동되면서 Task 시작 → ECR에서 이미지 pull 시도 → `ImagePullFailure` → 무한 재시도 → 배포 실패.
- ECR 먼저 생성하는 stage를 분리해 두면 최초 배포(boot-strap)와 이후 애플리케이션 업데이트의 종속성을 깔끔히 분리할 수 있다.

---
## 문제 6-2. 브랜치 전략 (3점)

### (a) 브랜치-환경 매핑 다이어그램

```
  feature/*  ──PR──▶  develop  ──PR──▶  staging  ──PR(manual approval)──▶  main
     │                    │                │                                │
     │                    ▼                ▼                                ▼
   (local)          development env    staging env                    production env
                    (auto deploy)      (auto deploy)            (deploy after approval)
```

### (b) feature/login → production 이벤트 시퀀스

1. 개발자가 `feature/login` 브랜치에서 작업 및 commit, `git push origin feature/login`.
2. GitHub에서 `feature/login → develop` **PR 생성**. CI(lint, test, build) 통과 후 리뷰어 approval, `Squash and merge`.
3. `develop` 브랜치 push 이벤트 → GitHub Actions 워크플로우 트리거 → **development 환경 자동 배포** (AWS ECS dev cluster).
4. QA팀이 development에서 기능 검증 완료 → `develop → staging` PR 생성, approval 후 merge.
5. `staging` 브랜치 push → **staging 환경 자동 배포** → 통합 테스트, 부하 테스트, 보안 스캔 수행.
6. staging 검증 완료 → `staging → main` PR 생성 (release PR). 릴리스 노트 작성.
7. **Environment Protection Rules**에 의해 main 브랜치는 수동 승인 필요. 승인자가 GitHub Actions UI에서 "Review deployments" → Approve.
8. main 브랜치 push 이벤트 → **production 배포 파이프라인** 실행 (ECR push → Terraform apply → Blue/Green 또는 Rolling deploy).
9. 배포 완료 후 CloudWatch 모니터링, 장애 시 즉시 rollback (Deployment Circuit Breaker가 자동 수행).

---
## 문제 6-3. ECR 로그인 + 빌드/푸시 YAML (4점)

In [ ]:
%%writefile ecr-push-steps.yml
- name: Login to Amazon ECR
  id: login-ecr
  uses: aws-actions/amazon-ecr-login@v2

- name: Build and Push Backend Image
  env:
    ECR_REGISTRY: ${{ steps.login-ecr.outputs.registry }}
    IMAGE_TAG: v${{ vars.IMAGE_TAG }}
    ENVIRONMENT: ${{ vars.ENVIRONMENT }}
  run: |
    docker build \
      -t "$ECR_REGISTRY/backend-$ENVIRONMENT:$IMAGE_TAG" \
      -t "$ECR_REGISTRY/backend-$ENVIRONMENT:latest" \
      -f ./backend/Dockerfile \
      ./backend
    docker push "$ECR_REGISTRY/backend-$ENVIRONMENT:$IMAGE_TAG"
    docker push "$ECR_REGISTRY/backend-$ENVIRONMENT:latest"

- name: Build and Push Frontend Image
  env:
    ECR_REGISTRY: ${{ steps.login-ecr.outputs.registry }}
    IMAGE_TAG: v${{ vars.IMAGE_TAG }}
    ENVIRONMENT: ${{ vars.ENVIRONMENT }}
  run: |
    docker build \
      -t "$ECR_REGISTRY/frontend-$ENVIRONMENT:$IMAGE_TAG" \
      -t "$ECR_REGISTRY/frontend-$ENVIRONMENT:latest" \
      -f ./frontend/Dockerfile \
      ./frontend
    docker push "$ECR_REGISTRY/frontend-$ENVIRONMENT:$IMAGE_TAG"
    docker push "$ECR_REGISTRY/frontend-$ENVIRONMENT:latest"

`[SCREENSHOT REQUIRED]` — GitHub Actions 실행 성공 화면 (`date && hostname` 출력 로그 포함)

---
## 문제 7-1. 아키텍처 의사결정 (ADR) (4점)

### 시나리오: 2인 스타트업, 일 100→10,000건, 월 예산 $200

### (a) 추천 플랫폼: **GCP Cloud Run**

**근거**:
- 2인 팀은 운영 부담 최소화가 생존 필수조건. Cloud Run은 ALB/cluster/taskDef 없이 `gcloud run deploy` 한 줄로 배포.
- 초기 일 100건(하루 수십 초만 CPU 점유)은 **scale-to-zero로 $5 미만** 달성 가능.
- 6개월 후 일 10,000건까지도 Cloud Run 단독으로 커버(월 $20~50). 조기에 ECS/K8s로 가면 operational overhead가 제품 개발 속도를 갉아먹음.

### (b) 리소스 구성

- **Cloud Run (backend)**: FastAPI 서비스, 2 vCPU / 2 GB / min=0, max=10
- **Cloud Run (frontend) 또는 Firebase Hosting**: nginx 또는 정적 SPA
- **Secret Manager**: `OPENAI_API_KEY`, DB 접속 정보
- **Artifact Registry**: 컨테이너 이미지 저장
- **Cloud Logging + Cloud Monitoring**: 자동 수집, 대시보드 1개 생성
- **(선택) Firestore**: 초기 유저 데이터 (관리형, 무료 쿼터 넉넉)
- **Workload Identity Federation**: GitHub Actions OIDC로 배포 (키리스)

### (c) 월간 비용 상세 (현재 vs 6개월 후)

**현재 — 일 100건 (월 3,000 요청)**
- Cloud Run 요청당 평균 500ms CPU, 512MB → 월 1,500 CPU-second × $0.000024 + 1,500 GB-sec × $0.0000025 ≈ **$0.04**
- Egress 무시할 수 있음, Logging 무료 쿼터 내
- **소계 ≈ $1/월** (무료 쿼터 차감 후 거의 0)

**6개월 후 — 일 10,000건 (월 300,000 요청)**
- 300,000 req × 0.5s = 150,000 CPU-sec × $0.000024 = $3.60 (2 vCPU면 $7.20)
- 150,000 GB-sec × $0.0000025 = $0.38
- OpenAI API 호출 비용(별도): 월 $100~$150 정도로 예상
- **Cloud Run 소계 ≈ $10~15/월** (OpenAI 제외)

**비교 — AWS ECS Fargate (0.5 vCPU, 1 GB, 상시 1 task)**
- Task: 0.5 × 0.04048 × 720 + 1.0 × 0.004445 × 720 = $14.58 + $3.20 = **$17.78/월** (항상)
- ALB: **$16.44/월** (LB hours) + 데이터 처리 비용
- **AWS 상시 비용 ≈ $35~40/월** (트래픽 무관)

**결론**: 저트래픽에서 Cloud Run이 압도적으로 저렴. 6개월 후에도 Cloud Run이 비용 우위. 예산 $200은 OpenAI API 비용까지 포함해도 여유.

### (d) 마이그레이션 / 스케일업 플랜

| 구간 | 전략 |
|------|------|
| **단기 (0~3개월)** | Cloud Run 단일 서비스, Firestore/SQLite 경량 DB, Cloud Logging 기본 설정, min=0 유지 |
| **중기 (3~12개월)** | Cloud SQL PostgreSQL 도입 (관리형), Cloud Monitoring 알람 설정, min=1(유료 $10~30)로 cold start 제거, VPC connector로 private 네트워크 분리, 정기 부하 테스트 |
| **장기 (12개월+)** | 일 트래픽 > 10만 건 또는 특수 요구사항(GPU, VPC peering 복잡) 시 GKE Autopilot 또는 AWS ECS Fargate로 이전 검토. 멀티 리전(asia-northeast3 + us-central1)으로 DR, Cloud CDN으로 정적 자산 가속 |

---
## 문제 7-2. 크로스 클라우드 시크릿 관리 (4점)

In [ ]:
%%writefile secrets.tf
# Part A: AWS ECS + Secrets Manager

# 1. Secrets Manager에서 기존 시크릿 참조
data "aws_secretsmanager_secret" "openai_api_key" {
  name = "openai-api-key-${var.environment}"
}

# 2. ECS Task Definition에서 valueFrom으로 주입
resource "aws_ecs_task_definition" "backend" {
  family                   = "backend-${var.environment}"
  network_mode             = "awsvpc"
  requires_compatibilities = ["FARGATE"]
  cpu                      = var.backend_cpu
  memory                   = var.backend_memory
  execution_role_arn       = aws_iam_role.ecs_task_execution.arn
  task_role_arn            = aws_iam_role.ecs_task.arn

  container_definitions = jsonencode([{
    name  = "backend"
    image = "${aws_ecr_repository.backend.repository_url}:v${var.image_tag}"
    portMappings = [{ containerPort = 8000 }]

    # 핵심: value가 아니라 valueFrom (Secrets Manager ARN)
    secrets = [
      {
        name      = "OPENAI_API_KEY"
        valueFrom = data.aws_secretsmanager_secret.openai_api_key.arn
      }
    ]

    logConfiguration = {
      logDriver = "awslogs"
      options = {
        awslogs-group         = aws_cloudwatch_log_group.backend.name
        awslogs-region        = var.aws_region
        awslogs-stream-prefix = "ecs"
      }
    }
  }])
}

# 3. Task Execution Role에 secrets 읽기 권한 부여 (필수)
resource "aws_iam_role_policy" "ecs_secrets_read" {
  role = aws_iam_role.ecs_task_execution.id
  policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Effect = "Allow"
      Action = ["secretsmanager:GetSecretValue"]
      Resource = data.aws_secretsmanager_secret.openai_api_key.arn
    }]
  })
}

In [ ]:
# Part B: GCP Cloud Run + Secret Manager

gcloud_command = """
# 1. Secret 생성 (최초 1회)
echo -n "sk-abc123..." | gcloud secrets create openai-api-key --data-file=-

# 2. Cloud Run 서비스 계정에 secretAccessor 권한 부여
gcloud secrets add-iam-policy-binding openai-api-key \\
  --member="serviceAccount:backend-sa@my-project.iam.gserviceaccount.com" \\
  --role="roles/secretmanager.secretAccessor"

# 3. Cloud Run 배포 시 --set-secrets로 주입 (환경변수 OPENAI_API_KEY로 노출)
gcloud run deploy backend \\
  --image=asia-northeast3-docker.pkg.dev/my-project/backend/backend:v1 \\
  --region=asia-northeast3 \\
  --service-account=backend-sa@my-project.iam.gserviceaccount.com \\
  --set-secrets=OPENAI_API_KEY=openai-api-key:latest \\
  --min-instances=0 --max-instances=10 \\
  --cpu=2 --memory=2Gi \\
  --allow-unauthenticated
"""
print(gcloud_command)

### Part C: WIF가 JSON 서비스 계정 키보다 안전한 이유 3가지

1. **Short-lived token (유출 내성)** — WIF는 GitHub Actions OIDC 토큰을 GCP STS와 교환해 **최대 1시간짜리 액세스 토큰**을 발급한다. 로그에 유출되거나 탈취되어도 자동 만료되어 장기 오용이 불가능하다. 반면 JSON 키는 영구 유효하여 한 번 유출되면 수동 revoke 전까지 무제한 사용 가능.

2. **저장할 시크릿 자체가 없음** — WIF는 CI 환경(GitHub Secrets, Vault 등)에 **어떤 크레덴셜도 저장하지 않는다**. 서비스 계정 email과 workload identity pool 이름만 있으면 되며, 이들은 노출되어도 인증에 사용 불가. JSON 키 방식은 필연적으로 GitHub Secrets에 평문 키를 저장해야 하고 이는 supply chain 공격 표면이 된다.

3. **세밀한 audit & 조건부 접근** — WIF는 Workload Identity Pool Provider 설정에서 "특정 repo의 main 브랜치에서 온 OIDC 토큰만 허용" 같은 **attribute condition**(`attribute.repository == 'org/repo' && attribute.ref == 'refs/heads/main'`)을 강제할 수 있다. Cloud Audit Logs에 어떤 repo/branch/workflow가 어떤 GCP API를 호출했는지 기록되어 추적성이 극대화된다. JSON 키는 "누가 어디서" 사용했는지 구분 불가능.

---
## 문제 8-1. OOM Kill 원인 및 해결 (4점)

### (a) 프로젝트 특화 OOM 원인 4가지

1. **LangChain/LangGraph 대화 히스토리 누적** — `ConversationBufferMemory`를 default로 쓰면 session마다 모든 message가 메모리에 유지됨. 수백 턴 대화에서 컨텍스트가 수십 MB~수백 MB로 성장하며, FastAPI가 여러 session을 동시에 보유하면 선형 증가.
2. **SSE 스트리밍 generator의 buffered chunks 미해제** — LLM 토큰을 `async for chunk in llm.astream(...)`으로 yield할 때 앞서 생성된 chunks를 list에 append하면서도 해제하지 않는 패턴이 자주 보임. 긴 응답(수천 토큰)마다 수 MB 누적.
3. **RAG 벡터 DB의 인메모리 임베딩** — ChromaDB/FAISS의 in-process 모드는 전체 임베딩 매트릭스(문서 10만 건 × 1536 dim × 4 byte ≈ 600MB)를 RAM에 로드. 512MB 메모리 할당으로는 OOM 직격.
4. **uvicorn worker의 동시 요청 버퍼링** — `--workers 2` 설정 시 각 worker가 독립 프로세스로 LangChain 그래프·토크나이저·임베딩을 중복 로드 → 2x 메모리. 여기에 각 worker가 동시에 N개 요청을 처리하면 LLM 응답 buffer도 N배 누적.

### (b) 각 원인별 진단

In [ ]:
# 원인 1, 2: CloudWatch Container Insights에서 메모리 증가 추이 확인
!aws cloudwatch get-metric-statistics \
  --namespace ECS/ContainerInsights \
  --metric-name MemoryUtilized \
  --dimensions Name=ClusterName,Value=agent-cluster-prod Name=ServiceName,Value=backend-service-prod \
  --start-time $(date -u -d '2 hours ago' +%FT%TZ) \
  --end-time   $(date -u +%FT%TZ) \
  --period 60 --statistics Maximum Average

# 원인 3: 컨테이너 내부에서 벡터 DB 크기와 RSS 비교 (ECS Exec)
!aws ecs execute-command --cluster agent-cluster-prod --task <TASK_ID> \
  --container backend --interactive --command "/bin/sh -c 'du -sh /app/chroma_db && cat /proc/self/status | grep VmRSS'"

# 원인 4: Python 메모리 프로파일링 (tracemalloc을 앱 코드에 삽입)
# import tracemalloc
# tracemalloc.start()
# ... 요청 처리 ...
# snapshot = tracemalloc.take_snapshot()
# for stat in snapshot.statistics('lineno')[:10]: print(stat)

### (c) 단기/중기/장기 해결

| 구분 | 해결 | 상세 |
|------|------|------|
| **단기** | Task Definition memory 2배 증설 | 1024 → 2048 (또는 4096), Terraform `backend_memory` 변수 변경 후 즉시 배포. 근본 원인은 그대로지만 장애 중단 방지 |
| **중기** | 대화 히스토리 외부화 + streaming generator 리팩터링 | LangChain MemoryType을 Redis로 변경 (`RedisChatMessageHistory`), ElastiCache Redis cluster 추가. SSE generator에서 buffer 누적 없도록 코드 점검, `del chunk` 명시. |
| **장기** | RAG 벡터 DB 분리 + 컨테이너 resource 전략 개편 | ChromaDB → **OpenSearch Service** 또는 **Pinecone** 같은 managed vector DB로 분리. backend 컨테이너는 검색 쿼리만 수행. Fargate task 크기도 CPU/memory 비율을 워크로드 측정값에 맞춰 재설계 |

### (d) Terraform 변경

In [ ]:
# terraform.tfvars 또는 main.tf
#
# 변경 전:  backend_cpu = "512", backend_memory = "1024"
# 변경 후:  backend_cpu = "1024", backend_memory = "4096"
#
# 값 선택 근거:
# - Fargate 유효 CPU/Memory 조합에서 1024 CPU는 2048/3072/4096/5120/6144/7168/8192 MB 지원.
# - 현재 1024 MB에서 반복적 OOM이므로 2배(2048)는 또 OOM 가능성, 4배(4096)로 안전 마진 확보.
# - Container Insights에서 peak MemoryUtilized를 실측하여 (peak × 1.5) 를 하한으로 잡는 것이 권장.
# - CPU도 512 → 1024로 상향 (Fargate는 CPU도 고정 조합이라 memory 4096을 쓰려면 최소 512 CPU, 이왕이면 1024로 LLM 응답 throughput 개선).

resource "aws_ecs_task_definition" "backend" {
  family = "backend-${var.environment}"
  cpu    = var.backend_cpu     # "1024"
  memory = var.backend_memory  # "4096"
  # ...
}

---
## 문제 8-2. 503 디버깅 (4점)

### (a) 첫 5분 — 확인 순서

1. **ALB Target Group에 healthy target이 있는가?** 대부분의 503은 타겟 부재. `aws elbv2 describe-target-health`로 즉시 확인.
2. **ECS Service의 RUNNING task 수가 desiredCount와 일치하는가?** desiredCount는 3인데 RUNNING 0이면 task 기동 자체가 실패 중.
3. **최근 5분간 CloudWatch Logs 에러**: backend 컨테이너의 stdout에 stack trace/crash 메시지. 503은 ALB가 업스트림을 못 찾아서 발생하므로 앱 크래시가 원인일 가능성.

### (b) Target 상태별 대응

| 상태 | 의미 | 대응 |
|------|------|------|
| `unhealthy` | 헬스체크 실패, 트래픽 제외 | 컨테이너 로그 확인, `/health` 엔드포인트 수동 호출, SG inbound 포트 확인, start_period 충분한지 검토 |
| `draining` | deregister 진행 중 (새 트래픽 x, 기존 요청만 완료) | deployment 중이면 정상. `deregistration_delay`가 너무 길면(>60s) 배포 속도 저하, 조정 검토 |
| `initial` | 등록됐지만 첫 헬스체크 완료 전 | 30s~1분 대기. 반복적으로 initial에 머물면 healthcheck threshold/timeout 확인 |
| `unused` | Target Group에 attach되지 않음 또는 Listener 미연결 | ALB Listener rule과 Target Group 연결 상태, `aws_lb_listener` HCL 확인 |

### (c) 디버깅 AWS CLI

In [ ]:
# 1. Target Group 헬스 확인 — 503의 1차 원인 후보
!aws elbv2 describe-target-health \
  --target-group-arn arn:aws:elasticloadbalancing:ap-northeast-2:123456789012:targetgroup/backend-tg-prod/abc123 \
  --query 'TargetHealthDescriptions[*].[Target.Id,TargetHealth.State,TargetHealth.Reason]' \
  --output table

# 2. ECS Service events — Task 기동 실패 원인 역추적
!aws ecs describe-services \
  --cluster agent-cluster-prod \
  --services backend-service-prod \
  --query 'services[0].events[0:10]' --output table

# 3. 최근 10분 컨테이너 로그 tail
!aws logs tail /ecs/backend-prod --since 10m --follow

# 4. RUNNING / PENDING / STOPPED task 수 확인
!aws ecs list-tasks --cluster agent-cluster-prod --service-name backend-service-prod --desired-status RUNNING
!aws ecs list-tasks --cluster agent-cluster-prod --service-name backend-service-prod --desired-status STOPPED

### (d) 503 의사결정 트리

```
503 Service Unavailable 발생
   │
   ▼
[Q1] ECS Service에 Running Task > 0 인가?
   │
   ├─ NO  → describe-services events 확인
   │         ├─ Resource 부족 ('Insufficient memory/CPU')     → Fargate capacity/limits 확인, cpu/memory 조정
   │         ├─ Image pull 실패 ('CannotPullContainerError')  → ECR 경로/태그 확인, executionRoleArn 권한 확인
   │         ├─ Task 크래시 ('Essential container exited')   → 로그 확인 → 앱 수정 후 재배포
   │         └─ SG/NetworkACL 문제 → subnet/SG outbound/NAT 검토
   │
   ▼ YES
[Q2] Target Group에 healthy target > 0 인가?
   │
   ├─ NO  → Target 상태 원인별 분기
   │         ├─ unhealthy  → /health 엔드포인트 수동 호출
   │         │                ├─ 200 OK → healthcheck path/port/matcher 설정 점검
   │         │                └─ 타임아웃 → 컨테이너 listen 포트(0.0.0.0:8000) + SG inbound 확인
   │         ├─ initial    → 대기 (start_period 내 정상)
   │         └─ unused     → ALB Listener/Target Group 연결 확인
   │
   ▼ YES
[Q3] ALB Listener Rule이 올바른 target group으로 라우팅하는가?
   │
   ├─ NO  → Rule priority, host/path pattern 확인, 콘솔에서 수정
   │
   ▼ YES
[Q4] 네트워크 경로 문제
   │
   └─ VPC Endpoint / NAT Gateway / Route Table 확인
       VPC Endpoint가 없어서 ECR/Secrets/CloudWatch 접근 불가한 경우 드물지 않음
```

---
## 문제 8-3. 배포 전략 + Circuit Breaker (4점)

### Part A: 3가지 배포 전략

**1. Rolling Update**
```
Before:       [v1][v1][v1]
Step 1:       [v1][v1][v2]    ← 한 대씩 교체
Step 2:       [v1][v2][v2]
Step 3:       [v2][v2][v2]
```
- **적합 시나리오**: 일반적인 무중단 업데이트, 리소스 제약 환경, 대부분의 평시 배포.
- **비용 오버헤드**: 배포 중 최대 ~33% 추가(`maximum_percent=200`일 때 일시적으로 v1 × 3 + v2 × 3 = 6대까지 허용 가능하지만 보통 +1대만). 약 10~15% 추가.

**2. Blue-Green**
```
Blue(v1):  [v1][v1][v1]  ← Production 트래픽 수신 중
Green(v2): [v2][v2][v2]  ← 병행 배포 (트래픽 x)
           ──────────────
ALB Listener가 Green으로 전환 (일괄 cut-over)
Blue는 대기 (즉시 rollback용)
           ──────────────
최종:      [v2][v2][v2]  ← Blue 제거
```
- **적합 시나리오**: 즉시 rollback이 중요, 무중단 강제, 스키마 변경 없는 대규모 릴리스, 크로스-팀 조율이 필요한 주요 버전 업.
- **비용 오버헤드**: 전환 구간에 **100% 추가 인프라** (Blue + Green 병행). 짧게는 수 분~수십 분.

**3. Canary**
```
[v1][v1][v1][v1] + [v2]               ← 5% 트래픽만 v2 (1/20)
[v1][v1][v1] + [v2][v2]               ← 20%
[v1][v1] + [v2][v2][v2]               ← 60%
                [v2][v2][v2][v2][v2]  ← 100%
```
- **적합 시나리오**: 사용자 영향 최소화, 실제 트래픽에서의 A/B 검증, 메트릭 기반 자동 롤아웃, ML 모델 가중치 변경 같은 민감한 릴리스.
- **비용 오버헤드**: 중간 단계에서 두 버전 공존 → 10~30% 추가. 롤아웃 기간이 길면 누적 증가.

### Part B: Circuit Breaker Terraform HCL

In [ ]:
%%writefile circuit-breaker.tf
resource "aws_ecs_service" "backend" {
  name            = "backend-service-${var.environment}"
  cluster         = aws_ecs_cluster.main.id
  task_definition = aws_ecs_task_definition.backend.arn
  desired_count   = var.backend_desired_count
  launch_type     = "FARGATE"

  # Deployment Circuit Breaker 활성화 + 자동 rollback
  deployment_circuit_breaker {
    enable   = true
    rollback = true
  }

  # Rolling update 설정 — 최대 200%, 최소 healthy 50%
  deployment_controller {
    type = "ECS"
  }

  deployment_maximum_percent         = 200
  deployment_minimum_healthy_percent = 50

  network_configuration {
    subnets          = var.private_subnet_ids
    security_groups  = [aws_security_group.backend_container.id]
    assign_public_ip = false
  }

  load_balancer {
    target_group_arn = aws_lb_target_group.backend.arn
    container_name   = "backend"
    container_port   = 8000
  }

  depends_on = [aws_lb_listener.backend]
}

### 서술: Circuit Breaker 없을 때 배포 실패 이벤트

- **t=0**: 운영자가 `terraform apply`로 새 Task Definition revision을 배포. ECS가 rolling update 시작.
- **t=30s**: 새 v2 Task 기동. 앱 초기화 중 `OPENAI_API_KEY` 누락으로 `raise ValueError(...)` → 컨테이너 exit 1.
- **t=1m**: Docker healthcheck 실패 3회 누적 → Task STOPPED. ECS Service가 desiredCount 유지를 위해 **같은 v2 revision으로 새 Task 기동**.
- **t=1m30s**: 두 번째 v2 Task도 동일 이유로 exit.
- **t=2m~**: ECS가 무한 재시도. 매 시도마다 ECR pull(이미지 사이즈 × 네트워크 비용), CloudWatch Logs에 에러 누적.
- **t=5m**: `deployment_minimum_healthy_percent=50`이라 v1 Task 일부는 살아있으나, rolling update 진행 중이라 v1 → v2 drain이 이미 시작되어 **v1 RUNNING 수도 감소** 중.
- **t=10m**: 최악의 경우 v1 RUNNING이 desiredCount의 50% 이하로 내려가면 ALB Target Group에 healthy target 부족 → **사용자 대면 503 발생**.
- **t=수십 분**: 운영자가 수동 개입 전까지 장애 지속. 비용은 실패 Task 실행 시간·ECR pull·로그 저장이 누적.

**Circuit Breaker가 있을 때**: 실패 Task가 일정 threshold(기본 10회)를 넘으면 ECS가 자동으로 배포를 중단하고 이전 revision으로 rollback을 트리거. 2~3분 내 v1으로 복귀 → 사용자 영향 0, 운영자는 알람만 받고 여유롭게 원인 조사 가능.

---
## 문제 9-1 Part A: 구조화 로깅 (1.5점)

In [ ]:
import json
import logging
import os
from datetime import datetime, timezone


class JSONFormatter(logging.Formatter):
    """구조화 JSON 로그 포매터.

    출력 필드: time, level, trace_id, message
    trace_id가 LogRecord extra에도 환경변수에도 없으면 'N/A'.
    """

    def format(self, record: logging.LogRecord) -> str:
        trace_id = (
            getattr(record, "trace_id", None)
            or os.environ.get("TRACE_ID")
            or "N/A"
        )

        payload = {
            "time": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "trace_id": trace_id,
            "message": record.getMessage(),
        }
        return json.dumps(payload, ensure_ascii=False)


def setup_logging(level: int = logging.INFO) -> None:
    handler = logging.StreamHandler()
    handler.setFormatter(JSONFormatter())

    root = logging.getLogger()
    root.handlers.clear()
    root.addHandler(handler)
    root.setLevel(level)


# 사용 예시
setup_logging()
logger = logging.getLogger(__name__)

logger.info("user login success", extra={"trace_id": "abc-123"})
logger.warning("no trace id here")  # trace_id: N/A
logger.error("openai api failed", extra={"trace_id": "xyz-999"})

---
## 문제 9-1 Part B: 멀티스테이지 Dockerfile (1.5점)

In [ ]:
%%writefile Dockerfile.multistage
# ===========================================================
# Stage 1: Builder — 의존성 설치만 담당 (최종 이미지에 포함 x)
# ===========================================================
FROM python:3.11-slim AS builder

WORKDIR /app

# uv 설치 (빌드 단계에서만 사용)
RUN pip install --no-cache-dir uv

# 의존성 파일만 먼저 복사 — 캐시 레이어 극대화
COPY pyproject.toml uv.lock ./

# production 의존성만 설치 (dev 패키지 제외)
RUN uv sync --frozen --no-dev


# ===========================================================
# Stage 2: Runtime — .venv만 복사한 슬림 이미지
# ===========================================================
FROM python:3.11-slim AS runtime

# healthcheck용 curl만 설치 (빌드 도구 없음)
RUN apt-get update && apt-get install -y --no-install-recommends curl \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# builder 스테이지에서 가상환경만 복사 — uv, pip 등 빌드 도구는 포함 안 됨
COPY --from=builder /app/.venv /app/.venv

# 소스 코드 복사 (의존성 이후에)
COPY . .

# venv 바이너리를 PATH에 추가 → uvicorn 직접 호출 가능
ENV PATH="/app/.venv/bin:$PATH" \
    PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

EXPOSE 8000

# --reload 제거 (production)
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

`[SCREENSHOT REQUIRED]` — `docker images | grep backend` 출력에서 기존 단일 스테이지(예: 850 MB) vs 멀티스테이지(예: 280 MB) 크기 비교 스크린샷

---
## 문제 9-2. 비용 최적화 (4점)

### 5개 전략 (현재 월 $500 → 목표 $300)

| 전략 | 변경 내용 | 예상 절감 (월) |
|------|----------|----------------|
| 1. **Fargate Spot (비프로덕션)** | dev/staging의 `capacity_provider_strategy`에 FARGATE_SPOT 100%, prod는 FARGATE on-demand 유지 | ~$80 (비프로덕션 Fargate 비용의 70% off) |
| 2. **CloudWatch Logs 보존 기간 단축** | 기본(never expire) → 14일 (비프로덕션) / 30일 (프로덕션). S3 Glacier로 장기 아카이브 | ~$30 (로그 스토리지 비용 대폭 감소) |
| 3. **ECR 라이프사이클 정책** | 각 repo당 최근 10개 이미지만 유지, untagged 이미지 7일 후 삭제 | ~$10 (수백 MB × 수백 개 → 수십 개로 감소) |
| 4. **Container Insights 비활성화 (dev 한정)** | 비프로덕션 cluster의 `containerInsights=disabled`, 프로덕션만 enabled | ~$20 (환경당 ~$10/월 절감) |
| 5. **비프로덕션 야간/주말 스케줄링** | EventBridge Scheduler로 평일 09~19시 외 `desired_count=0`. staging은 배포 트리거 시에만 기동 | ~$60 (비프로덕션 가동 시간 70% → 40%) |

**합계 절감 ≈ $200/월 → 목표 $300 달성**

### Terraform HCL 2개 이상

In [ ]:
%%writefile cost-optimization.tf
# ===========================================================
# 전략 1: Fargate Spot 도입 (비프로덕션)
# ===========================================================
resource "aws_ecs_cluster_capacity_providers" "main" {
  cluster_name = aws_ecs_cluster.main.name

  capacity_providers = ["FARGATE", "FARGATE_SPOT"]

  default_capacity_provider_strategy {
    capacity_provider = var.environment == "prod" ? "FARGATE" : "FARGATE_SPOT"
    weight            = 100
    base              = var.environment == "prod" ? 1 : 0
  }
}

resource "aws_ecs_service" "backend" {
  name            = "backend-service-${var.environment}"
  cluster         = aws_ecs_cluster.main.id
  task_definition = aws_ecs_task_definition.backend.arn
  desired_count   = var.backend_desired_count

  # prod는 FARGATE 100%, 그 외 환경은 FARGATE_SPOT 100%
  capacity_provider_strategy {
    capacity_provider = "FARGATE_SPOT"
    weight            = var.environment == "prod" ? 0 : 100
    base              = 0
  }

  capacity_provider_strategy {
    capacity_provider = "FARGATE"
    weight            = var.environment == "prod" ? 100 : 0
    base              = var.environment == "prod" ? 1 : 0
  }

  # ... 나머지 설정 생략 ...
}


# ===========================================================
# 전략 2: CloudWatch Logs 보존 기간 단축
# ===========================================================
resource "aws_cloudwatch_log_group" "backend" {
  name              = "/ecs/backend-${var.environment}"
  retention_in_days = var.environment == "prod" ? 30 : 14

  tags = {
    Environment = var.environment
    Purpose     = "Cost optimization — shorter retention"
  }
}


# ===========================================================
# 전략 3: ECR 라이프사이클 정책 (최근 10개만 유지)
# ===========================================================
resource "aws_ecr_repository" "backend" {
  name                 = "backend-${var.environment}"
  image_tag_mutability = "IMMUTABLE"

  image_scanning_configuration {
    scan_on_push = true
  }
}

resource "aws_ecr_lifecycle_policy" "backend" {
  repository = aws_ecr_repository.backend.name

  policy = jsonencode({
    rules = [
      {
        rulePriority = 1
        description  = "Keep last 10 tagged images"
        selection = {
          tagStatus     = "tagged"
          tagPrefixList = ["v"]
          countType     = "imageCountMoreThan"
          countNumber   = 10
        }
        action = { type = "expire" }
      },
      {
        rulePriority = 2
        description  = "Delete untagged images after 7 days"
        selection = {
          tagStatus   = "untagged"
          countType   = "sinceImagePushed"
          countUnit   = "days"
          countNumber = 7
        }
        action = { type = "expire" }
      }
    ]
  })
}


# ===========================================================
# 전략 4: Container Insights 선택적 활성화
# ===========================================================
resource "aws_ecs_cluster" "main" {
  name = "agent-cluster-${var.environment}"

  setting {
    name  = "containerInsights"
    value = var.environment == "prod" ? "enabled" : "disabled"
  }
}


# ===========================================================
# 전략 5: EventBridge Scheduler로 비프로덕션 야간 자동 축소
# ===========================================================
resource "aws_scheduler_schedule" "scale_down_night" {
  count = var.environment == "prod" ? 0 : 1

  name       = "ecs-scale-down-night-${var.environment}"
  group_name = "default"

  flexible_time_window { mode = "OFF" }

  # 평일 19:00 KST (10:00 UTC)에 desired_count=0
  schedule_expression          = "cron(0 10 ? * MON-FRI *)"
  schedule_expression_timezone = "UTC"

  target {
    arn      = "arn:aws:scheduler:::aws-sdk:ecs:updateService"
    role_arn = aws_iam_role.scheduler.arn

    input = jsonencode({
      Cluster      = aws_ecs_cluster.main.name
      Service      = aws_ecs_service.backend.name
      DesiredCount = 0
    })
  }
}

resource "aws_scheduler_schedule" "scale_up_morning" {
  count = var.environment == "prod" ? 0 : 1

  name       = "ecs-scale-up-morning-${var.environment}"
  group_name = "default"

  flexible_time_window { mode = "OFF" }

  # 평일 09:00 KST (00:00 UTC)에 desired_count=2
  schedule_expression          = "cron(0 0 ? * MON-FRI *)"
  schedule_expression_timezone = "UTC"

  target {
    arn      = "arn:aws:scheduler:::aws-sdk:ecs:updateService"
    role_arn = aws_iam_role.scheduler.arn

    input = jsonencode({
      Cluster      = aws_ecs_cluster.main.name
      Service      = aws_ecs_service.backend.name
      DesiredCount = 2
    })
  }
}

---

## 제출 체크리스트

- [ ] 1-1: Dockerfile + 라인 번호 참조 서술 + `docker build` 스크린샷
- [ ] 1-2: 수정된 docker-compose + 4개 오류 설명 + `docker compose up` healthy 스크린샷
- [ ] 1-3: 3단계 절차 + git 명령 예시 + filter-repo/bfg 언급
- [ ] 2-1: nginx.conf + `/api/` prefix 제거 동작 확인 스크린샷
- [ ] 2-2: 원인 3개 + 프로젝트 특정 디버깅 명령 + DNS 원리 서술
- [ ] 2-3: healthcheck YAML + 파라미터 표 + start_period=0 시나리오
- [ ] 3-1: Dual ALB 다이어그램 + Excalidraw 소스 + AWS ECS 콘솔 스크린샷
- [ ] 3-2: Task Def 3개 오류 + executionRole vs taskRole 3문장 서술
- [ ] 3-3: Health Check 3단계 표 + Terraform HCL + `/health` vs `/ready`
- [ ] 4-1: AWS vs GCP 8개 항목 비교 표 + CLI 예시
- [ ] 4-2: Cloud Run용 nginx.conf + HTTPS/Host/timeout 3가지 설명
- [ ] 4-3: Cold Start 5단계 + 전략 3개 + min-instances=1 조건
- [ ] 5-1: SG HCL 2개 + security_groups 참조 이유 서술
- [ ] 5-2: variables.tf (≥6개 + validation) + outputs.tf (≥3개) + default 없음 서술
- [ ] 5-3: 수동 문제점 3개 + Terraform 해결 + 4개 명령어 + tfvars 유효 조합
- [ ] 6-1: GitHub Actions 4개 오류 + 3-Stage 순서 서술
- [ ] 6-2: 브랜치-환경 다이어그램 + feature→prod 이벤트 시퀀스
- [ ] 6-3: ECR 로그인 + 빌드/푸시 YAML + 실행 성공 스크린샷
- [ ] 7-1: 추천 플랫폼 + 구성 + 비용 계산 + 마이그레이션 플랜
- [ ] 7-2: AWS Terraform + GCP gcloud + WIF 3가지 근거
- [ ] 8-1: 프로젝트 특화 OOM 원인 4개 + 진단 명령 + 해결 단/중/장기 + Terraform 변경
- [ ] 8-2: 우선순위 3개 + Target 상태 표 + CLI 4개 + 의사결정 트리
- [ ] 8-3: 3가지 배포 전략 + Circuit Breaker HCL + 없을 때 장애 시나리오
- [ ] 9-1: JSON 로깅 Formatter + 멀티스테이지 Dockerfile + 크기 비교 스크린샷
- [ ] 9-2: 5개 이상 비용 최적화 전략 + Terraform HCL 2개 이상 + $300 달성 합리성